<a href="https://colab.research.google.com/github/elbuh0/genai-course/blob/main/genai-course/week7/homework/Week_7_HW6_RAG_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HW 6 — Production RAG with Evaluation & Failure Audit
**COM SCI-X 455 · Week 7 · Megan Armani, PhD**

> Push to `genai-course/week7/`

---

## Setup

In [1]:
   !pip install -U langchain langchain-core langchain-openai langchain-community langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers pypdf --quiet

In [2]:
import os, json
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('openai')

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter
from openai import OpenAI

client = OpenAI()
llm    = ChatOpenAI(model="gpt-4o", temperature=0)
print("Ready")

/tmp/ipykernel_1391/3451156040.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Ready


## Part 1 — Build FAISS Index on Synthetic Data Generation for Military CV Training Applications


In [3]:
# Document knowledge base: diffusion-based synthetic data for defense-relevant UAV CV (Nov 2024 – Nov 2026)
DOCUMENTS = [
    "https://pure.uva.nl/ws/files/263445059/136790V.pdf",                     # 1 Snel et al. (TNO), SPIE 2025 – diffusion inpainting, UAV vehicle detection
    "https://arxiv.org/pdf/2609.02556",                                       # 2 Eker et al. (TNO), 2026 – diffusion RGB→IR, UAV vehicle detection
    "https://arxiv.org/pdf/2512.13869",                                       # 3 CFHA (UMich + Army Research Lab), 2025 – diffusion Sim2Real, UAV human detection
    "https://arxiv.org/pdf/2512.12508",                                       # 4 Generative Spatiotemporal Aug. (UMich + ARL), 2025 – video diffusion, UAV
    "https://cctech.org.ua/images/docs/Articles/2026/paper_26_2_6.pdf",       # 5 Gen-Thermal-UAV, 2026 – video diffusion, counter-UAS thermal
    "https://openaccess.thecvf.com/content/CVPR2026/papers/Li_Visual_Prototype_Conditioned_Focal_Region_Generation_for_UAV-Based_Object_Detection_CVPR_2026_paper.pdf",  # 6 UAVGen, CVPR 2026 (defense framing weak)
    "https://openaccess.thecvf.com/content/ICCV2025/papers/Fang_Adapting_Vehicle_Detectors_for_Aerial_Imagery_to_Unseen_Domains_with_ICCV_2025_paper.pdf",  # 7 AGenDA (CMU + ARL), ICCV 2025 – manned aerial orthophotos
    "https://arxiv.org/pdf/2607.02718",                                       # 8 Panev et al. (CMU + ARL), 2026 – aerial orthophotos, diagnostic use
    "https://arxiv.org/pdf/2604.18076",                                       # 9 Fokkinga et al. (TNO), 2026 – military vehicles, ground-level
    "https://arxiv.org/pdf/2609.02299",                                       # 10 Stein et al. (TNO), 2026 – camouflaged military vehicles, ground-level
]

assert len(DOCUMENTS) >= 10, "Add at least 10 documents before proceeding"

In [4]:
# URL test

import requests

headers = {"User-Agent": "Mozilla/5.0"}
pdf_bytes = {}

for url in DOCUMENTS:
    r = requests.get(url, headers=headers, timeout=60)
    ok = r.status_code == 200 and r.content[:4] == b"%PDF"
    print("OK  " if ok else "FAIL", r.status_code, url[:80])
    if ok:
        pdf_bytes[url] = r.content

print(f"\n{len(pdf_bytes)}/{len(DOCUMENTS)} valid PDFs")

OK   200 https://pure.uva.nl/ws/files/263445059/136790V.pdf
OK   200 https://arxiv.org/pdf/2609.02556
OK   200 https://arxiv.org/pdf/2512.13869
OK   200 https://arxiv.org/pdf/2512.12508
OK   200 https://cctech.org.ua/images/docs/Articles/2026/paper_26_2_6.pdf
OK   200 https://openaccess.thecvf.com/content/CVPR2026/papers/Li_Visual_Prototype_Condit
OK   200 https://openaccess.thecvf.com/content/ICCV2025/papers/Fang_Adapting_Vehicle_Dete
OK   200 https://arxiv.org/pdf/2607.02718
OK   200 https://arxiv.org/pdf/2604.18076
OK   200 https://arxiv.org/pdf/2609.02299

10/10 valid PDFs


In [5]:
# Extract PDFs and split into chunks (chunk_size=400-512, chunk_overlap=50).
# Build a FAISS index with HuggingFaceEmbeddings('all-MiniLM-L6-v2').
# Save with vectorstore.save_local('mil_cv_index').
# Print: 'Index built with N chunks'
# Code source: Lab 6 with LLM assistance

!pip install pypdf --quiet
import io, re, requests
from pypdf import PdfReader

TITLES = [
    "Snel et al. 2025 (TNO) - diffusion inpainting for UAV vehicle detection",
    "Eker et al. 2026 (TNO) - diffusion RGB-to-IR translation for UAV vehicle detection",
    "CFHA 2025 (UMich/ARL) - diffusion Sim2Real for UAV human detection",
    "Generative Spatiotemporal Augmentation 2025 (UMich/ARL) - video diffusion for UAV detection",
    "Gen-Thermal-UAV 2026 - video diffusion thermal benchmark for counter-UAS",
    "UAVGen CVPR 2026 - visual-prototype diffusion for UAV object detection",
    "AGenDA ICCV 2025 (CMU/ARL) - fine-tuned diffusion for aerial vehicle detection",
    "Panev et al. 2026 (CMU/ARL) - generative diagnosis of aerial vehicle detectors",
    "Fokkinga et al. 2026 (TNO) - class-specific diffusion for military vehicle detection",
    "Stein et al. 2026 (TNO) - diffusion camouflage editing for military vehicle detection",
]
assert len(TITLES) == len(DOCUMENTS)

def pdf_text(url):
    r = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    text = "\n".join(p.extract_text() or "" for p in PdfReader(io.BytesIO(r.content)).pages)
    idx = max(text.rfind("\nReferences"), text.rfind("\nREFERENCES"))
    if idx > len(text) * 0.5: text = text[:idx]                # drop bibliography
    text = re.sub(r"-\n(\w)", r"\1", text)                     # rejoin hyphenated words
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)               # join wrapped lines
    return text

TEXTS = [pdf_text(u) for u in DOCUMENTS]
for t, txt in zip(TITLES, TEXTS): print(f"{len(txt):>7} chars  {t[:60]}")

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50, separators=["\n\n", "\n", ". ", " ", ""])
docs = splitter.create_documents(TEXTS, metadatas=[{"source": i} for i in range(len(TEXTS))])
for d in docs: d.page_content = f"[{TITLES[d.metadata['source']]}]\n{d.page_content}"

# Build FAISS index
embeddings = HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2')
vectorstore = FAISS.from_documents(docs, embeddings)
vectorstore.save_local('mil_cv_index') # persist for reuse
retriever = vectorstore.as_retriever(search_kwargs={'k': 5})
print(f"Index built with {len(docs)} chunks")

  38406 chars  Snel et al. 2025 (TNO) - diffusion inpainting for UAV vehicl
  46233 chars  Eker et al. 2026 (TNO) - diffusion RGB-to-IR translation for
  35737 chars  CFHA 2025 (UMich/ARL) - diffusion Sim2Real for UAV human det
  65150 chars  Generative Spatiotemporal Augmentation 2025 (UMich/ARL) - vi
  33444 chars  Gen-Thermal-UAV 2026 - video diffusion thermal benchmark for
  33807 chars  UAVGen CVPR 2026 - visual-prototype diffusion for UAV object
  36483 chars  AGenDA ICCV 2025 (CMU/ARL) - fine-tuned diffusion for aerial
  35886 chars  Panev et al. 2026 (CMU/ARL) - generative diagnosis of aerial
  35929 chars  Fokkinga et al. 2026 (TNO) - class-specific diffusion for mi
  41108 chars  Stein et al. 2026 (TNO) - diffusion camouflage editing for m


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (5 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Index built with 1305 chunks


In [6]:
# Test retrieval against the index (k=5); for each query, print the top chunk returned.

import textwrap

test_qs = [
    "How much did Stable Diffusion 3.5 with ControlNet improve infrared vehicle detection mAP on the Kust4K and VTUAV datasets?",
    "At what ratio of synthetic to real data did adding more diffusion-generated images stop improving or start degrading detection?",
]

for test_q in test_qs:
    retrieved = retriever.invoke(test_q)

    print(f"\n{'='*80}\nRetrieved {len(retrieved)} chunks for: {test_q}")

    for i, doc in enumerate(retrieved):
        print(f"\nChunk {i+1}:")
        wrapped_text = textwrap.fill(doc.page_content, width=80)
        print(wrapped_text)


Retrieved 5 chunks for: How much did Stable Diffusion 3.5 with ControlNet improve infrared vehicle detection mAP on the Kust4K and VTUAV datasets?

Chunk 1:
[Eker et al. 2026 (TNO) - diffusion RGB-to-IR translation for UAV vehicle
detection] . The best-performing approach, Stable Diffusion 3.5 with ControlNet,
improves mAP from 50.8 to 60.1 on Kust4K and from 25.6 to 38.4 on VTUA V,
compared to detectors trained only on sourcedomain infrared data. Further gains
are observed on VTUA V by increasing output diversity through multiple diffusion
seeds (+1.1 mAP) and prompt variations (+3.3 mAP)

Chunk 2:
[Eker et al. 2026 (TNO) - diffusion RGB-to-IR translation for UAV vehicle
detection] . Without any target-domain data, performance is substantially lower
on VTUA V (25.6 mAP) than on Kust4K (50.8 mAP). Adding the real infrared target-
domain training data yields large improvements on both datasets, increasing
performance to 67.5 and 52.3 mAP, respectively. Adding target-domain RGB images
i

In [7]:
# RAG Chain + Reranker + Faithfulness Gate

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel, RunnableLambda
from sentence_transformers import CrossEncoder
import json

RAG_SYSTEM = """You are an R&D analyst at a defense technology company, reviewing academic research on diffusion-based synthetic data for training computer-vision models on UAV and military imagery.
Use ONLY the context below to answer. Do not use prior knowledge.
Answer using relevant information explicitly stated in the context.
If the answer is explicitly stated anywhere in the context, provide it.
Only say 'Not found in provided documents' if the information is genuinely absent.
Do not append 'Not found in provided documents' if you have already provided an answer.

Domain rules:
- Attribute every number (mAP, dataset size, augmentation ratio) to the study it comes from; never merge figures from different studies into one claim.
- State the imagery type when it matters: UAV, manned-aerial orthophoto, ground-level, or thermal/infrared.
- Do not generalize one study's result to all UAV or military detection.
- If the question contains a false or unsupported premise, do not answer 'Not found'; state what the documents actually say on that topic and correct the premise.
- Report research findings only; do not give operational, targeting, or procurement recommendations."""

RAG_TEMPLATE = RAG_SYSTEM + """

Context:
{context}

Question: {question}
Answer (cite the bracketed study title for each claim):"""

prompt = PromptTemplate.from_template(RAG_TEMPLATE)
llm = ChatOpenAI(model="gpt-4o", temperature=0)

def fmt(docs): return "\n\n".join(f"[{i+1}] {d.page_content}" for i, d in enumerate(docs))

# Two-stage retrieval: wide FAISS pool + cross-encoder reranking
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def smart_retrieve(q, k=5, pool=50):
    cands = vectorstore.similarity_search(q, k=pool)
    scores = reranker.predict([(q, d.page_content) for d in cands])
    return [d for _, d in sorted(zip(scores, cands), key=lambda x: -x[0])][:k]

smart_retriever = RunnableLambda(smart_retrieve)

rag_chain = (
    {"context": smart_retriever | fmt, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# Variant returning context + question + answer, so the gate judges the exact context used
rag_chain_ctx = RunnableParallel(
    context=smart_retriever | fmt, question=RunnablePassthrough()
).assign(answer=prompt | llm | StrOutputParser())

# Faithfulness gate
def check_faithfulness(question: str, answer: str, context: str) -> dict:
    result = llm.bind(response_format={"type": "json_object"}).invoke(f"""Rate the faithfulness of this answer from 1-5 using ONLY the context.
Do not use prior knowledge or judge whether the context is plausible.
If the answer says the information is not found and the context lacks it, score 5.
5 = fully supported by context, 1 = contradicts or ignores context.
Question: {question}
Context: {context}
Answer: {answer}
Return JSON: {{"score": int, "reason": str}}""")
    out = json.loads(result.content)
    out["flag"] = "pass" if out["score"] >= 4 else "fail"
    return out

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [8]:
import textwrap
import pandas as pd

# 20-question suite (one fact per question): (type, question, reference answer, expected source index, key string)
# Source indices follow DOCUMENTS/TITLES order: 0 Snel, 1 Eker, 2 CFHA, 3 Spatiotemporal, 4 Gen-Thermal-UAV,
# 5 UAVGen, 6 AGenDA, 7 Panev, 8 Fokkinga, 9 Stein
SUITE = [
    # --- Positive: answerable, one per paper ---
    ("positive", "In the study that turned normal color drone images into thermal (infrared) images, which method worked best?",
     "Stable Diffusion 3.5 with ControlNet.", 1, "Stable Diffusion 3.5"),
    ("positive", "In the study that pasted extra vehicles into drone images, which AI tool did they use?",
     "AnyDoor (a diffusion-based inpainting model).", 0, "AnyDoor"),
    ("positive", "When researchers had only 8 real photos of each type of military vehicle, how much did adding AI-generated images help detection?",
     "Up to +8.0 mAP50 with 8 real samples (real+FLUX vs real only).", 8, "8.0"),
    ("positive", "In the study that used AI to add camouflage to military vehicle photos, what mix of AI-made and real images worked best?",
     "25% synthetic / 75% real.", 9, "75% real"),
    ("positive", "What are the three steps of the CFHA method for making computer-generated drone images look more realistic?",
     "Global Style Transfer, Local Refinement, and Hallucination Removal.", 2, "Hallucination Removal"),
    ("positive", "What is the main AI model inside UAVGen called?",
     "Visual Prototype Conditioned Diffusion Model (VPC-DM).", 5, "Visual Prototype Conditioned Diffusion"),
    ("positive", "In the study that used AI video generation to create extra drone training images, which video tool was used for the drone images?",
     "LTX-Video-13B.", 3, "LTX-Video"),
    ("positive", "Where were the two new aerial photo datasets in the AGenDA study taken?",
     "Selwyn, New Zealand (LINZ) and Utah, USA (UGRC).", 6, "Utah"),
    ("positive", "In the study that used AI-generated aerial images to find weaknesses in vehicle detectors, which image generator was chosen?",
     "Imagen 3.", 7, "Imagen 3"),
    ("positive", "How many videos are in the Gen-Thermal-UAV dataset?",
     "220 videos.", 4, "220"),

    # --- Negative: not in documents, should return 'Not found' ---
    ("negative", "How many AI-generated images were used to train the U.S. Army's Project Maven?",
     "Not found in provided documents.", None, None),
    ("negative", "How fast do UAVGen-trained detectors run on a small onboard drone computer like an NVIDIA Jetson?",
     "Not found in provided documents.", None, None),
    ("negative", "How big do drones appear on radar in these studies?",
     "Not found in provided documents.", None, None),
    ("negative", "Did AI-generated radar (SAR) images help detect ships?",
     "Not found in provided documents.", None, None),
    ("negative", "When was the MQ-9 Reaper drone first used?",
     "Not found in provided documents.", None, None),

    # --- Adversarial: prior-knowledge conflict or misleading framing ---
    ("adversarial", "Since AI-generated images are better than real photos, how much better did training on only AI images do in the military vehicle study?",
     "False premise: FLUX-only underperformed real-only (48.8 vs 62.9 mAP50 with 8 samples; 57.7 vs 83.1 with 24).", 8, "underperforms"),
    ("adversarial", "Why did using only AI-generated images work best in the camouflage study?",
     "False premise: 25% synthetic / 75% real was best; 100% synthetic degraded performance.", 9, "75% real"),
    ("adversarial", "Which drone was used to film the images in the AGenDA datasets?",
     "False premise: they are aerial orthophoto tiles at 12.5 cm GSD from LINZ and UGRC platforms; no drone is stated.", 6, "12.5 cm"),
    ("adversarial", "Covering up newly revealed areas in AI-generated video frames is essential, so how much did it help in the video-generation drone study?",
     "False premise: masking gave no meaningful improvement; disocclusion handling can be safely ignored.", 3, "safely ignored"),
    ("adversarial", "Stable Diffusion is the best tool for making aerial images, so how well did it do in the study on finding detector weaknesses?",
     "Contradicts premise: open-source SD 1.5/XL/3.5 often produced unrealistic perspectives or weak prompt adherence; Imagen 3 was chosen.", 7, "unrealistic perspectives"),
]

# Retrieval quality: 5 = key fact in top chunk, 4 = key fact in top k, 3 = right paper only, 1 = wrong papers; None for negatives
def retrieval_score(docs, src, key):
    if src is None: return None
    for rank, d in enumerate(docs):
        if d.metadata["source"] == src and key.lower() in d.page_content.lower():
            return 5 if rank == 0 else 4
    return 3 if any(d.metadata["source"] == src for d in docs) else 1

# Answer quality + correctness: LLM judge against the reference answer
def judge_answer(q, answer, reference, qtype):
    r = llm.bind(response_format={"type": "json_object"}).invoke(f"""Grade a RAG answer against a reference answer.
Question type: {qtype} (positive = answerable; negative = should say 'Not found in provided documents'; adversarial = should reject the false premise using the documents).
Question: {q}
Reference answer: {reference}
Answer: {answer}
answer_quality 1-5: 5 = complete and precise, 1 = wrong or unhelpful.
correct = true only if the key facts match the reference (negative: says not found; adversarial: rejects the false premise).
Return JSON: {{"answer_quality": int, "correct": bool, "reason": str}}""")
    return json.loads(r.content)

rows = []
for i, (qtype, q, reference, src, key) in enumerate(SUITE, 1):
    docs = smart_retrieve(q)
    context = fmt(docs)
    answer = (prompt | llm | StrOutputParser()).invoke({"context": context, "question": q})
    faith = check_faithfulness(q, answer, context)
    grade = judge_answer(q, answer, reference, qtype)
    if "not found in provided documents" in answer.lower() and qtype != "negative":   # exact refusal phrase only
        grade.update(correct=False, answer_quality=1)
    rows.append({"#": i, "type": qtype, "question": q,
                 "retrieval": retrieval_score(docs, src, key),
                 "answer_quality": grade["answer_quality"],
                 "faithfulness": faith["score"],
                 "correct": grade["correct"],
                 "answer": answer, "judge_reason": grade["reason"]})
    print(f"{i}. [{qtype}] {textwrap.fill(q, width=90)}")
    print("   A: " + textwrap.fill(answer, width=90, subsequent_indent="      "))
    print(f"   Retrieval: {rows[-1]['retrieval']}  Answer: {grade['answer_quality']}  "
          f"Faithfulness: {faith['score']}  Correct: {grade['correct']}\n")

df = pd.DataFrame(rows)
display(df[["#", "type", "question", "retrieval", "answer_quality", "faithfulness", "correct"]])
print(df.groupby("type")[["retrieval", "answer_quality", "faithfulness", "correct"]].mean().round(2))
df.to_csv("eval_results_v3.csv", index=False)

1. [positive] In the study that turned normal color drone images into thermal (infrared) images, which
method worked best?
   A: The study by Eker et al. 2026 (TNO) mentions that the FLUX-LoRA method preserves the
      visibility of dark vehicles particularly well in lowlight conditions, suggesting it
      worked effectively for RGB-to-IR translation in those scenarios (Eker et al. 2026
      (TNO) - diffusion RGB-to-IR translation for UAV vehicle detection).
   Retrieval: 3  Answer: 1  Faithfulness: 5  Correct: False

2. [positive] In the study that pasted extra vehicles into drone images, which AI tool did they use?
   A: In the study that pasted extra vehicles into drone images, they used the zero-shot SD1.4
      text-to-image diffusion model (Snel et al. 2025 (TNO) - diffusion inpainting for UAV
      vehicle detection).
   Retrieval: 4  Answer: 1  Faithfulness: 5  Correct: False

3. [positive] When researchers had only 8 real photos of each type of military vehicle, how much di

,#,type,question,retrieval,answer_quality,faithfulness,correct
0,1,positive,In the study that turned normal color drone im...,3.0,1,5,False
1,2,positive,In the study that pasted extra vehicles into d...,4.0,1,5,False
2,3,positive,When researchers had only 8 real photos of eac...,4.0,5,5,True
3,4,positive,In the study that used AI to add camouflage to...,3.0,4,5,True
4,5,positive,What are the three steps of the CFHA method fo...,4.0,5,5,True
5,6,positive,What is the main AI model inside UAVGen called?,5.0,5,5,True
6,7,positive,In the study that used AI video generation to ...,4.0,5,5,True
7,8,positive,Where were the two new aerial photo datasets i...,5.0,5,5,True
8,9,positive,In the study that used AI-generated aerial ima...,4.0,4,5,True
9,10,positive,How many videos are in the Gen-Thermal-UAV dat...,5.0,5,5,True


             retrieval  answer_quality  faithfulness  correct
type                                                         
adversarial        3.4             4.2           5.0      0.8
negative           NaN             5.0           5.0      1.0
positive           4.1             4.0           5.0      0.8


## Part 2 — Failure Mode Audit

In [9]:
# Failure audit: rule-based classification + type-specific LLM diagnosis
import re, json, textwrap
import pandas as pd
pd.set_option("display.max_colwidth", None)

# Failure type definitions
FAILURE_TYPES = {
    "A": "Retrieval failure - the right document was in the index but not retrieved",
    "B": "Faithfulness failure - the answer was generated from training knowledge, not the context",
    "C": "Format failure - the answer didn't meet the format constraint in the system prompt",
    "D": "Over-refusal - the model said 'Not found' for a question that was answerable",
}
print("FAILURE TYPE DEFINITIONS")
for k, v in FAILURE_TYPES.items(): print(f"  Type {k}: {v}")

REFUSAL = "not found in provided documents"
STUDY_NAMES = [t.split(" ")[0] for t in TITLES]          # citation markers: Snel, Eker, CFHA, ...
OVERRIDES = {9: False}                                    # manual judge corrections: {question #: correct}
ALL_CHUNKS = list(vectorstore.docstore._dict.values())    # every indexed chunk (for locating the answer chunk)

def J(prompt):                                            # JSON-mode LLM call
    return json.loads(llm.bind(response_format={"type": "json_object"}).invoke(prompt).content)

# Evidence helpers
def answer_chunk(src, key):
    """The indexed chunk that holds the answer (exhaustive search), or None."""
    return next((d for d in ALL_CHUNKS if d.metadata["source"] == src and key.lower() in d.page_content.lower()), None)

def faiss_rank(q, src, key, pool=50):
    for r, d in enumerate(vectorstore.similarity_search(q, k=pool)):
        if d.metadata["source"] == src and key.lower() in d.page_content.lower():
            return r
    return ">50"

def ungrounded_numbers(answer, context):
    return sorted(n for n in set(re.findall(r"\d+(?:\.\d+)?", answer)) if n not in context)

# Rule-based classification
# A: answer chunk not in retrieved context (refusal caused by a retrieval miss also counts as A)
# B: faithfulness < 4, or numbers in the answer absent from the context
# C: non-refusal answer without a study citation (the system prompt's citation rule)
# D: refusal although the answer chunk WAS in the retrieved context
def classify(qtype, src, key, answer, faith, docs, context, q):
    refused = REFUSAL in answer.lower()
    ev = {"refused": refused, "cited": refused or any(n in answer for n in STUDY_NAMES),
          "ungrounded_numbers": ungrounded_numbers(answer, context), "faithfulness": faith}
    if src is not None:
        ev["key_in_context"] = any(d.metadata["source"] == src and key.lower() in d.page_content.lower() for d in docs)
        ev["faiss_rank"] = faiss_rank(q, src, key)
    types = []
    if qtype == "negative":
        if not refused:
            types.append("B - Faithfulness (answered unanswerable question)"
                         if faith < 4 or ev["ungrounded_numbers"] else "Other - Over-answering")
    else:
        sub = "vocabulary gap" if ev.get("faiss_rank") == ">50" else f"ranking (FAISS rank {ev.get('faiss_rank')})"
        if refused:
            types.append("D - Over-refusal" if ev.get("key_in_context") else f"A - Retrieval, {sub} (surfaced as refusal)")
        elif src is not None and not ev["key_in_context"]:
            types.append(f"A - Retrieval, {sub}")
        if not refused and (faith < 4 or ev["ungrounded_numbers"]):
            types.append("B - Faithfulness")
        if not ev["cited"]:
            types.append("C - Format (no study citation)")
        if not types:
            types.append("Other - Grounded but wrong (selection/attribution)")
    return types, ev

# Type-specific diagnosis prompts
def diagnose_A(q, reference, docs, src, key, ev):
    retrieved = "\n".join(f"[{i+1}] {d.page_content[:350]}" for i, d in enumerate(docs))
    correct = answer_chunk(src, key)
    return J(f"""You are auditing a Type A failure ({FAILURE_TYPES['A']}). In this audit, the chunk containing the answer was not retrieved.
Question: {q}
Reference answer: {reference}
Evidence: {json.dumps(ev)}  (faiss_rank = position of the answer chunk in raw vector search; ">50" means it was not in the top 50)

RETRIEVED CHUNKS (what the model saw):
{retrieved}

CORRECT ANSWER CHUNK (in the index, not retrieved):
{correct.page_content if correct else "Could not be located by key string."}

Tasks:
1. why_wrong_chunks: Examine the retrieved chunks against the correct chunk. Explain specifically why the wrong chunks outranked it: compare the question's wording with the correct chunk's wording (vocabulary mismatch), topic overlap of sibling chunks from the same paper, generic intro/abstract text matching broad questions, or the answer fact being split across chunk boundaries.
2. fix_category: exactly one of "chunk size", "query phrasing", "k", "embedding model".
3. fix: a specific, concrete change (e.g. an exact chunk size, an exact rewritten query, an exact k value, or a named embedding model) and why it would surface the correct chunk.
Return JSON: {{"why_wrong_chunks": str, "fix_category": str, "fix": str}}""")

def diagnose_B(q, answer, context):
    return J(f"""You are auditing a Type B failure ({FAILURE_TYPES['B']}).
Question: {q}

RETRIEVED CONTEXT:
{context}

MODEL ANSWER:
{answer}

Tasks:
1. unsupported_claim: quote VERBATIM the shortest exact substring of the model answer that the context does not support.
2. likely_source: where the claim probably came from (training knowledge, a misread of a nearby sentence, a different study in the context, merging two studies).
3. prompt_modification: the exact rule sentence(s) to add to the system prompt that would prevent this class of error.
Return JSON: {{"unsupported_claim": str, "likely_source": str, "prompt_modification": str}}""")

def diagnose_D(q, reference, context):
    return J(f"""You are auditing a Type D failure ({FAILURE_TYPES['D']}). The evidence for the answer was in the retrieved context.
Question: {q}
Reference answer: {reference}

RETRIEVED CONTEXT:
{context}

Tasks:
1. evidence_sentence: quote the sentence(s) in the context that would have allowed an answer.
2. difficulty_cause: exactly one of "ambiguous phrasing", "context window limits", "too-broad question", "other".
3. explanation: what made the question difficult to answer from this context, and why the model refused instead.
4. fix: a generalizable fix (pipeline or prompt rule, not a rewrite of this one question).
Return JSON: {{"evidence_sentence": str, "difficulty_cause": str, "explanation": str, "fix": str}}""")

def diagnose_other(q, reference, answer, types, ev, context):
    return J(f"""You are auditing a failed RAG answer that fits none of these types: {json.dumps(FAILURE_TYPES)}
Rule-assigned label(s): {types}. Evidence: {json.dumps(ev)}
Question: {q}
Reference answer: {reference}
Model answer: {answer}
Retrieved context (truncated): {context[:3000]}
In 1-2 sentences each, give the specific root cause and a generalizable pipeline-level fix.
Return JSON: {{"root_cause": str, "fix": str}}""")

# Audit loop
def audit(results, suite, overrides=OVERRIDES):
    rows = []
    for _, r in results.iterrows():
        if overrides.get(r["#"], r["correct"]):
            continue
        qtype, q, reference, src, key = suite[r["#"] - 1]
        docs = smart_retrieve(q)                    # deterministic: same context the eval saw
        context = fmt(docs)
        types, ev = classify(qtype, src, key, r["answer"], r["faithfulness"], docs, context, q)
        primary = types[0][0]                       # A / B / C / D / O(ther)
        if primary == "A":
            d = diagnose_A(q, reference, docs, src, key, ev)
            cause, fix = d["why_wrong_chunks"], f"[{d['fix_category']}] {d['fix']}"
        elif primary == "B":
            d = diagnose_B(q, r["answer"], context)
            cause, fix = f"Unsupported: \"{d['unsupported_claim']}\" ({d['likely_source']})", d["prompt_modification"]
        elif primary == "D":
            d = diagnose_D(q, reference, context)
            cause, fix = f"[{d['difficulty_cause']}] {d['explanation']}", d["fix"]
        else:
            d = diagnose_other(q, reference, r["answer"], types, ev, context)
            cause, fix = d["root_cause"], d["fix"]
        rows.append({"Q": r["#"], "Question": q, "Failure Type": "; ".join(types), "Primary": primary,
                     "Definition": FAILURE_TYPES.get(primary, "Other - fits none of Types A-D"),
                     "Root Cause": cause, "Proposed Fix": fix, "Evidence": json.dumps(ev),
                     "Answer": r["answer"], "Context": context, "Detail": d})
    return pd.DataFrame(rows)

audit_df = audit(df, SUITE)

# Failure classification table
display(audit_df[["Q", "Question", "Failure Type", "Definition", "Root Cause", "Proposed Fix"]])

uncited = [r["#"] for _, r in df.iterrows()
           if REFUSAL not in r["answer"].lower() and not any(n in r["answer"] for n in STUDY_NAMES)]
print(f"\nType C scan - uncited non-refusal answers (all 20): {uncited or 'none'}")
print("\nFailure counts by primary type:\n", audit_df["Primary"].value_counts().to_string())

# Detailed reports per failure
W = lambda s, ind="             ": textwrap.fill(str(s), 90, subsequent_indent=ind)
for _, r in audit_df.iterrows():
    d = r["Detail"]
    print("\n" + "=" * 90 + f"\nQ{r['Q']}  [{r['Failure Type']}]\nDefinition: {r['Definition']}\n{W(r['Question'], '')}\n" + "-" * 90)
    if r["Primary"] == "A":
        print("Retrieved chunks:")
        for line in r["Context"].split("\n\n"):
            print("  " + textwrap.shorten(line, 150))
        print("Why wrong:    " + W(d["why_wrong_chunks"]))
        print(f"Fix [{d['fix_category']}]: " + W(d["fix"]))
    elif r["Primary"] == "B":
        print("Retrieved context:\n" + "\n".join("  " + textwrap.shorten(l, 200) for l in r["Context"].split("\n\n")))
        claim = d["unsupported_claim"]
        hl = r["Answer"].replace(claim, f">>> {claim} <<<") if claim in r["Answer"] else r["Answer"] + f"\n  (claim not found verbatim: {claim})"
        print("Answer (unsupported claim highlighted):\n  " + W(hl, "  "))
        print("Likely source: " + W(d["likely_source"]))
        print("Prompt fix:    " + W(d["prompt_modification"]))
    elif r["Primary"] == "D":
        print("Evidence in context: " + W(d["evidence_sentence"]))
        print(f"Difficulty cause:    {d['difficulty_cause']}")
        print("Explanation:         " + W(d["explanation"]))
        print("Fix:                 " + W(d["fix"]))
    else:
        print("Root cause: " + W(d["root_cause"]))
        print("Fix:        " + W(d["fix"]))

audit_df.drop(columns=["Detail"]).to_csv("failure_audit_general.csv", index=False)

FAILURE TYPE DEFINITIONS
  Type A: Retrieval failure - the right document was in the index but not retrieved
  Type B: Faithfulness failure - the answer was generated from training knowledge, not the context
  Type C: Format failure - the answer didn't meet the format constraint in the system prompt
  Type D: Over-refusal - the model said 'Not found' for a question that was answerable


,Q,Question,Failure Type,Definition,Root Cause,Proposed Fix
0,1,"In the study that turned normal color drone images into thermal (infrared) images, which method worked best?","A - Retrieval, vocabulary gap",Retrieval failure - the right document was in the index but not retrieved,"The retrieved chunks did not contain the specific method 'Stable Diffusion 3.5 with ControlNet' mentioned in the correct answer. The retrieved chunks focused on general methods and datasets related to RGB-to-IR translation but did not specifically address the best-performing method. This indicates a vocabulary mismatch between the question and the retrieved chunks, as the specific method name was not present in the retrieved chunks. Additionally, the retrieved chunks may have had topic overlap with the question but lacked the specific detail required to answer it correctly.","[embedding model] Use a more advanced embedding model that can better capture specific method names and technical terms, such as 'Stable Diffusion 3.5 with ControlNet', to improve retrieval accuracy for questions requiring specific technical details."
1,2,"In the study that pasted extra vehicles into drone images, which AI tool did they use?",Other - Grounded but wrong (selection/attribution),Other - fits none of Types A-D,"The model incorrectly attributed the AI tool used in the study to the zero-shot SD1.4 text-to-image diffusion model instead of the correct AnyDoor diffusion-based inpainting model. This misattribution likely occurred due to the presence of multiple diffusion models mentioned in the retrieved context, leading to confusion.","Enhance the model's ability to disambiguate between similar entities or tools mentioned in the context by incorporating more sophisticated entity recognition and disambiguation techniques. Additionally, improve the model's understanding of context-specific cues that can help accurately attribute information to the correct source."
2,9,"In the study that used AI-generated aerial images to find weaknesses in vehicle detectors, which image generator was chosen?",Other - Grounded but wrong (selection/attribution),Other - fits none of Types A-D,"The model answer included multiple image generators used in the study, rather than identifying the specific one chosen for the task, leading to a misattribution of the primary image generator used.","Implement a more precise selection mechanism in the answer generation process to ensure that when multiple options are presented in the context, the model accurately identifies and selects the specific option that directly answers the question."
3,18,Which drone was used to film the images in the AGenDA datasets?,D - Over-refusal,Over-refusal - the model said 'Not found' for a question that was answerable,"[other] The question asks specifically about the drone used to film the images in the AGenDA datasets. However, the retrieved context does not mention any specific drone being used. Instead, it mentions that the images are aerial orthophoto tiles from LINZ and UGRC platforms. The model correctly identified that there was no information about a specific drone, leading to the 'Not found' response.","Implement a rule to recognize when a question is asking for specific details not present in the context, and provide a response indicating that the context does not contain the requested specific information, rather than a generic 'Not found'."



Type C scan - uncited non-refusal answers (all 20): none

Failure counts by primary type:
 Primary
O    2
A    1
D    1

Q1  [A - Retrieval, vocabulary gap]
Definition: Retrieval failure - the right document was in the index but not retrieved
In the study that turned normal color drone images into thermal (infrared) images, which
method worked best?
------------------------------------------------------------------------------------------
Retrieved chunks:
  [1] [Eker et al. 2026 (TNO) - diffusion RGB-to-IR translation for UAV vehicle detection] . The model is trained exclusively on paired RGB–IR [...]
  [2] [Eker et al. 2026 (TNO) - diffusion RGB-to-IR translation for UAV vehicle detection] . The default augmentation setup for infrared drone [...]
  [3] [Gen-Thermal-UAV 2026 - video diffusion thermal benchmark for counter-UAS] . For drone detection itself, Svanström et al. compiled a multi- [...]
  [4] [Eker et al. 2026 (TNO) - diffusion RGB-to-IR translation for UAV vehicle detectio

Part 2 Analysis: Most Common Failure Mode and Why

Of the four failure modes, Type A (retrieval) was the most common. There were at least two reasons, i.e.: (1) plain-language user questions that did not match the technical language of the studies in the chunks, potentially addressable with an intermediary LLM that translates plain-English queries into more technically precise language (at the cost of an additional risk of distorted user query); and (2) answer chunks that ranked outside the retrieved chunks, which a reranker partially fixed. Chunk sizes too small for the longer sentences common in technical and academic papers may also contribute, but this was not verified (in the interest of time, an LLM provides possible fixes for each failure in this implementation, and for chunk size it proposed a fix of smaller chunks, that contradicted its own diagnosis of facts being split across chunks).

The other most common mode was an "Other" bucket of failures to properly interpret information in context, for example returning the wrong model as the one decided upon in a study when the retrieved chunks included multiple models that were tested but ultimately discarded. Separately, one Type D over-refusal occurred on a misleading-framing question that suggested a drone was used to take images when none had been used; the model answered 'Not found' instead of correcting the premise.

The architectural change that would most improve accuracy would be improving retrieval: deeper reranked retrieval to address answer chunks being outranked, potentially larger chunks if testing confirms facts are being split, and potentially an LLM intermediary that would translate plain-language queries into more technically precise language that would pair better with the content of the chunks.

## Part 3 - Multi-Turn with Memory Strategy

In [10]:
# Multi-turn RAG pipeline with three memory strategies
import json, textwrap, tiktoken
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Prerequisite check: these come from the index cell and Cell 1 (RAG chain + reranker + gate)
missing = [n for n in ["vectorstore", "RAG_SYSTEM", "llm", "fmt", "smart_retrieve", "check_faithfulness"]
           if n not in globals()]
assert not missing, f"Run the index cell and Cell 1 first. Missing: {missing}"

enc = tiktoken.encoding_for_model("gpt-4o")
def ntok(s): return len(enc.encode(s))

MT_TEMPLATE = RAG_SYSTEM + """
Use the conversation memory only to understand what the user is referring to (e.g. "that method", "those studies"). Take all facts from the Context.

Conversation memory:
{memory}

Context:
{context}

Question: {question}
Answer (cite the bracketed study title for each claim):"""
mt_chain = PromptTemplate.from_template(MT_TEMPLATE) | llm | StrOutputParser()

# Resolve follow-up references so retrieval gets a standalone question
def standalone(question, memory):
    if not memory.strip(): return question
    return llm.invoke(f"""Rewrite the follow-up question as a standalone question, replacing references like "that method", "those studies" or "it" with what they refer to in the conversation memory.
Keep the user's own wording otherwise; do not add technical terms. Return only the question.
Conversation memory:
{memory}
Follow-up question: {question}""").content.strip()

# Strategy 1: full history (questions + answers, verbatim)
class BufferMemory:
    name = "full_history"
    def __init__(self): self.turns = []
    def render(self): return "\n\n".join(f"User: {q}\nAssistant: {a}" for q, a in self.turns)
    def update(self, q, a): self.turns.append((q, a))

# Strategy 2: summary compression (progressive summary, LangChain-style prompt)
class SummaryMemory:
    name = "summary"
    def __init__(self): self.summary = ""
    def render(self): return self.summary
    def update(self, q, a):
        self.summary = llm.invoke(f"""Progressively summarize the conversation, adding onto the previous summary and returning a new summary of at most 120 words.
Current summary:
{self.summary}
New lines of conversation:
User: {q}
Assistant: {a}
New summary:""").content.strip()

# Strategy 3: slot tracking (structured state, updated each turn)
class SlotMemory:
    name = "slots"
    def __init__(self): self.slots = {"user_goal": None, "constraints": [], "current_focus": None, "key_findings": []}
    def render(self): return json.dumps(self.slots, indent=1) if self.slots["user_goal"] else ""
    def update(self, q, a):
        r = llm.bind(response_format={"type": "json_object"}).invoke(f"""Update this conversation state JSON with the new exchange. Keep existing entries unless the user changed them.
- user_goal: the user's overall research question
- constraints: scope filters the user has added (e.g. military, drone imagery, thermal)
- current_focus: the specific study, method or result the user is now asking about
- key_findings: short facts from the answers, each with its study name and any numbers
Current state: {json.dumps(self.slots)}
User: {q}
Assistant: {a}
Return the full updated JSON with the same keys.""")
        self.slots = json.loads(r.content)

def run_conversation(memory, questions):
    log = []
    for t, q in enumerate(questions, 1):
        mem = memory.render()
        sq = standalone(q, mem)
        docs = smart_retrieve(sq)
        context = fmt(docs)
        a = mt_chain.invoke({"memory": mem or "(none)", "context": context, "question": q})
        faith = check_faithfulness(sq, a, context)
        memory.update(q, a)
        log.append({"strategy": memory.name, "turn": t, "question": q, "standalone_query": sq,
                    "memory_tokens": ntok(mem), "faithfulness": faith["score"], "answer": a})
    return log

print("Cell A ready: BufferMemory, SummaryMemory, SlotMemory, run_conversation")

Cell A ready: BufferMemory, SummaryMemory, SlotMemory, run_conversation


In [11]:
import pandas as pd, re, textwrap

CONVERSATION = [
    "Does AI-generated (synthetic) training data improve military drone computer-vision models?",
    "Which of these studies actually used images taken from drones?",
    "Of those, which ones worked with thermal (infrared) images?",
    "In the study that turned normal drone images into thermal ones, which method worked best?",
    "How much did that method improve vehicle detection?",
    "What limitations did the authors mention for that approach?",
]

# Reference keys for turns with one known answer (Eker et al.); any match counts
KEYS = {4: ["Stable Diffusion 3.5", "SD 3.5", "SD3.5"],
        5: ["60.1", "38.4"],
        6: ["engine heat", "unobservable", "gap"]}

# Facts whose survival in memory we track (any alias counts)
TRACKED = {
    "Study name (Eker et al.)":         ["Eker"],
    "Method (SD 3.5 + ControlNet)":     ["ControlNet", "Stable Diffusion 3.5", "SD 3.5"],
    "Result numbers (60.1 / 38.4)":     ["60.1", "38.4"],
    "Constraint: drone/UAV imagery":    ["drone", "UAV"],
    "Constraint: thermal/infrared":     ["thermal", "infrared"],
    "Constraint: military":             ["military"],
    "Limitation (gap to real IR data)": ["gap", "engine heat", "unobservable"],
}
# Standalone query for turns 5-6 must name the method or study for 'that method/approach' to count as resolved
RESOLVE = ["ControlNet", "Stable Diffusion", "SD 3.5", "Eker", "RGB-to-IR", "thermal", "infrared"]

def has(text, aliases): return any(a.lower() in text.lower() for a in aliases)
def snippet(text, aliases, w=70):
    for a in aliases:
        m = re.search(re.escape(a), text, re.I)
        if m: return "..." + text[max(0, m.start()-w): m.end()+w].replace("\n", " ") + "..."
    return ""

# --- Run the same conversation through all three strategies ---
all_logs, memories = [], {}
for Mem in (BufferMemory, SummaryMemory, SlotMemory):
    mem = Mem()
    log = run_conversation(mem, CONVERSATION)
    all_logs += log
    memories[mem.name] = mem
    print(f"\n{'='*90}\nSTRATEGY: {mem.name}")
    for r in log:
        print(f"\nT{r['turn']} Q: {r['question']}")
        print(f"   Standalone: {r['standalone_query']}")
        print("   A: " + textwrap.fill(r["answer"], 90, subsequent_indent="      "))
        print(f"   Memory tokens: {r['memory_tokens']}  Faithfulness: {r['faithfulness']}")
    print(f"\nFinal memory ({mem.name}):\n{mem.render()}")

mt = pd.DataFrame(all_logs)
mt["key_hit"] = mt.apply(lambda r: has(r["answer"], KEYS[r["turn"]]) if r["turn"] in KEYS else None, axis=1)
mt["resolved"] = mt.apply(lambda r: has(r["standalone_query"], RESOLVE) if r["turn"] in (5, 6) else None, axis=1)

# --- Per-turn view ---
display(mt.pivot(index="turn", columns="strategy", values=["memory_tokens", "faithfulness", "key_hit"]))

# --- Retention: was each fact introduced in an answer, and does it survive in final memory? ---
ret_rows, examples = [], []
for name, mem in memories.items():
    final = mem.render()
    answers = mt[mt.strategy == name].sort_values("turn")
    for fact, aliases in TRACKED.items():
        intro = next((int(r.turn) for r in answers.itertuples() if has(r.answer, aliases)), None)
        if intro is None:
            status = "never introduced"
        else:
            status = "retained" if has(final, aliases) else "LOST"
        ret_rows.append({"strategy": name, "fact": fact, "status": status})
        if status != "never introduced":
            examples.append({"strategy": name, "fact": fact, "status": status, "introduced_turn": intro,
                             "evidence": snippet(final, aliases) if status == "retained"
                                         else "absent from final memory; was in turn-" + str(intro) + " answer: "
                                              + snippet(answers[answers.turn == intro].answer.iloc[0], aliases)})
ret = pd.DataFrame(ret_rows)
display(ret.pivot(index="fact", columns="strategy", values="status"))

# --- Strategy comparison table ---
summary = []
for name, mem in memories.items():
    s = mt[mt.strategy == name]
    r = ret[ret.strategy == name]
    introduced = r[r.status != "never introduced"]
    summary.append({
        "strategy": name,
        "final memory tokens": ntok(mem.render()),
        "mean memory tokens / turn": round(s.memory_tokens.mean()),
        "mean faithfulness": round(s.faithfulness.mean(), 2),
        "reference answers hit (T4-6)": f"{int(s.key_hit.sum())}/3",
        "'that method' resolved (T5-6)": f"{int(s.resolved.sum())}/2",
        "facts retained": f"{(introduced.status == 'retained').sum()}/{len(introduced)}",
        "facts lost": ", ".join(introduced[introduced.status == "LOST"].fact) or "none",
    })
display(pd.DataFrame(summary).set_index("strategy"))

# --- Specific examples: retained vs lost, per strategy ---
print("\n" + "=" * 90 + "\nRETAINED / LOST EXAMPLES PER STRATEGY\n" + "=" * 90)
ex = pd.DataFrame(examples)
for name in memories:
    print(f"\n[{name}]")
    for _, e in ex[ex.strategy == name].iterrows():
        print(f"  {e['status']:<9} {e['fact']} (introduced T{e['introduced_turn']})")
        print("            " + textwrap.fill(e["evidence"], 85, subsequent_indent="            "))

mt.to_csv("multiturn_results.csv", index=False)
ex.to_csv("multiturn_retention_examples.csv", index=False)


STRATEGY: full_history

T1 Q: Does AI-generated (synthetic) training data improve military drone computer-vision models?
   Standalone: Does AI-generated (synthetic) training data improve military drone computer-vision models?
   A: Yes, AI-generated (synthetic) training data can improve military drone computer-vision
      models. According to Fokkinga et al. 2026 (TNO) - class-specific diffusion for
      military vehicle detection, generative AI-based synthetic data effectively supports
      military object detection in low-data settings. The study found that FLUX-based data
      consistently improves performance when combined with real images, and adding
      structural guidance via ControlNet provides additional benefits when limited real
      samples are available.
   Memory tokens: 0  Faithfulness: 5

T2 Q: Which of these studies actually used images taken from drones?
   Standalone: Which of the studies mentioned, such as the one by Fokkinga et al. 2026 (TNO) on class-spec

memory_tokens               faithfulness                    key_hit  \
strategy  full_history slots summary full_history slots summary full_history   
turn                                                                           
1                    0     0       0            5     5       5         None   
2                  115   152      83            5     5       5         None   
3                  135   282     140            5     5       5         None   
4                  255   291     174            5     5       5        False   
5                  375   362     169            5     5       5        False   
6                  428   352     198            5     5       5         True   

                        
strategy slots summary  
turn                    
1         None    None  
2         None    None  
3         None    None  
4         True    True  
5         True    True  
6         True    True

strategy,full_history,slots,summary
fact,,,
Constraint: drone/UAV imagery,retained,retained,retained
Constraint: military,retained,retained,retained
Constraint: thermal/infrared,retained,retained,LOST
Limitation (gap to real IR data),retained,retained,retained
Method (SD 3.5 + ControlNet),retained,retained,retained
Result numbers (60.1 / 38.4),never introduced,retained,LOST
Study name (Eker et al.),retained,retained,retained


,final memory tokens,mean memory tokens / turn,mean faithfulness,reference answers hit (T4-6),'that method' resolved (T5-6),facts retained,facts lost
strategy,,,,,,,
full_history,595,218,5.0,1/3,2/2,6/6,none
summary,170,127,5.0,3/3,1/2,5/7,"Result numbers (60.1 / 38.4), Constraint: thermal/infrared"
slots,385,240,5.0,3/3,0/2,7/7,none



RETAINED / LOST EXAMPLES PER STRATEGY

[full_history]
  retained  Study name (Eker et al.) (introduced T3)
            ...ose, which ones worked with thermal (infrared) images? Assistant: 1. [Eker et al.
            2026 (TNO) - diffusion RGB-to-IR translation for UAV vehicle d...
  retained  Method (SD 3.5 + ControlNet) (introduced T1)
            ...ce when combined with real images, and adding structural guidance via ControlNet
            provides additional benefits when limited real samples are available....
  retained  Constraint: drone/UAV imagery (introduced T1)
            ...User: Does AI-generated (synthetic) training data improve military drone computer-
            vision models? Assistant: Yes, AI-generated (synthetic) trai...
  retained  Constraint: thermal/infrared (introduced T3)
            ...t found in provided documents  User: Of those, which ones worked with thermal
            (infrared) images? Assistant: 1. [Eker et al. 2026 (TNO) - diffusion ...
  retained  

Part 3 Recommendation Memo: Full-History Injection

We recommend full-history injection for the appropriate multi-turn memory strategy for the task of novice R&D analyst exploring a corpus of technical documents in order to identify the feasibility of a given avenue of investigation. In this scenario, the conversation may take multiple turns, pivoting from an initial query upon indication that a given line of inquiry is well-trodden ground, or conversely probing further to identify novel research areas. In this scenario, where a summary strategy may fail to carry forward important details of a user’s multi-path discussion, or where a slot-type memory strategy would be less-well-suited given an undefined and exploratory task type, full history provides maximum context and information from varied or increasingly in-depth questions, all at an acceptable compute cost for a small 6-turn session. However, for a much larger multi-turn task, unbounded session growth could be an issue. In this case, switching to a summary model that was appropriately prompted to preserver key study details would be the better strategy.
